# 04 · Encoder (E1): fine-tuned XLM-R base · GPU

Long texts: training uses one random 512-token window per text and epoch; prediction averages the class probabilities of evenly spaced windows covering the text (up to `max_chunks` per language) and also stores min/max/std over windows.

Ways to run:
- local GPU machine: `uv run python -m hw3.encoder` (same code, same config), then open this notebook from the report cells;
- Colab / Kaggle: run this notebook; the first cell clones the repo. On Kaggle add the competition as input data and enable internet; on Colab set Kaggle credentials for the download;
- `SMOKE = True`: tiny CPU run (48 train texts, 64-token windows) that only checks the pipeline; outputs go to `artifacts/smoke/`.

After a cloud run, download the zip from the last cell and extract it into `artifacts/` of the local repo.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kerfmanner/ai-text-detection-hw3-ucu-ai.git"
CLOUD = "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if CLOUD:
    if not Path("ai-text-detection-hw3-ucu-ai").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("ai-text-detection-hw3-ucu-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.24.0"], check=True)
    sys.path.insert(0, str(Path("src").resolve()))
CLOUD

In [ ]:
SMOKE = False

import torch

from hw3 import config, encoder
from hw3.data import download
from hw3.runs import compare, export_outputs, has_preds, report, val_report
from hw3.metrics import by_group

config.set_smoke(SMOKE)
download()
cfg = config.load_config("encoder")
torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", cfg

In [ ]:
encoder.run(cfg)

In [ ]:
rep = report("E1")
rep["figure"].savefig(config.ROOT / "images" / "confusion_E1.png", dpi=150, bbox_inches="tight")
rep["per_class"]

Validation macro-F1 by number of windows the text was split into (long-text handling).

In [ ]:
r = val_report("E1")
by_group(r.assign(windows=r.n_chunks.clip(upper=5)), "windows")

Comparison with the baseline on the same validation split (B0 column appears when its predictions are present).

In [ ]:
compare(["B0", "E1"])

In [ ]:
export_outputs("E1_outputs", ["preds/E1_*", "runs/E1*", "logs/E1*"])